# BPE の実装

Python の標準ライブラリだけで、Byte Pair Encoding（BPE）による学習とトークン化を行う実装です。

- **学習**：コーパス（学習用のテキストの集合）から、頻繁に隣り合うトークンのペアを繰り返しまとめ、マージ規則を得る
- **トークン化**：学習したマージ規則を順に適用して、テキストをサブワードに分割する
- マージ回数を変えて、分割の粒度を比較する

トークンはテキストを扱うときの単位、サブワードは単語より短い部分も扱える分割単位です。まず3文の小さなコーパスで、文字単位のトークンをまとめる手順を確かめます。その後、少し文を増やしてマージ回数と分割結果の関係を観察します。

## コーパスの準備

3文をコーパスとして使います。ファイルに依存せず、この小さな例でペアの集計とマージを1つずつ確かめます。

In [1]:
corpus = [
    "吾輩は猫である",
    "猫と吾輩",
    "吾輩猫です",
]

['吾輩は猫である', '猫と吾輩', '吾輩猫です']

BPE は最初にコーパスを**文字単位**に分割した状態からスタートします。この文字の集まりが初期語彙になります。

In [2]:
# コーパスを文字単位に分割したものが、学習の出発点になる
token_lists = [list(text) for text in corpus]
token_lists

[['吾', '輩', 'は', '猫', 'で', 'あ', 'る'],
 ['猫', 'と', '吾', '輩'],
 ['吾', '輩', '猫', 'で', 'す']]

各文が 1 文字ずつのトークン列になりました。ここから「隣り合う頻出ペアをまとめる」操作を繰り返していきます。

## 隣接ペアを数える

学習の 1 ステップ目は、トークン列の中で**隣り合うトークンのペア**をすべて数えることです。`Counter` にペアを投げ込むだけで頻度表が得られます。`zip(tokens, tokens[1:])` で「隣り合う 2 つ組」を順に取り出せる点がポイントです。

In [3]:
from collections import Counter


def count_pairs(token_lists: list[list[str]]) -> Counter[tuple[str, str]]:
    """トークン列の集合から、隣接するトークンペアの出現頻度を数える。"""
    counter: Counter[tuple[str, str]] = Counter()
    for tokens in token_lists:
        for pair in zip(tokens, tokens[1:]):
            counter[pair] += 1
    return counter


count_pairs(token_lists).most_common(5)

[(('吾', '輩'), 3),
 (('猫', 'で'), 2),
 (('輩', 'は'), 1),
 (('は', '猫'), 1),
 (('で', 'あ'), 1)]

## ペアをマージする

最頻出ペアが決まったら、トークン列の中に現れるそのペアを、連結した 1 つのトークンに置き換えます。左から順に走査し、該当ペアを見つけたら 2 トークンぶんまとめて 1 トークンにし、インデックスを 2 つ進めます。

In [4]:
def merge_pair(tokens: list[str], pair: tuple[str, str]) -> list[str]:
    """トークン列中の該当する隣接ペアを、連結した 1 トークンに置き換える。"""
    merged = "".join(pair)
    new_tokens: list[str] = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(merged)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

# 「吾輩は猫である」の文字列に (吾, 輩) をマージしてみる
merge_pair(list("吾輩は猫である"), ("吾", "輩"))

['吾輩', 'は', '猫', 'で', 'あ', 'る']

先頭の `吾` と `輩` が 1 つの `吾輩` にまとまり、残りはそのままです。この `count_pairs`（数える）と `merge_pair`（まとめる）の 2 つを繰り返すだけで、BPE の学習ができます。

## 学習ループ

2 つの部品を組み合わせて、学習の繰り返しを実装します。

学習結果は `BPETokenizer` というクラスのオブジェクトにまとめます。中身はマージ規則列 `merge_rules`（どのペアをどの順でまとめたか）と、獲得した語彙 `vocab` です。

In [5]:
from dataclasses import dataclass


@dataclass
class BPETokenizer:
    """学習済み BPE トークナイザ。マージ規則列と語彙を保持し、encode で分割する。"""

    merge_rules: list[tuple[str, str]]
    vocab: set[str]

    def encode(self, text: str) -> list[str]:
        """テキストを文字に分割し、マージ規則を先頭から順に適用して分割する。"""
        tokens = list(text)
        for pair in self.merge_rules:
            tokens = merge_pair(tokens, pair)
        return tokens

入れ物ができました。次に、2 つの部品（`count_pairs` と `merge_pair`）を`num_merges` 回まわす学習ループを書きます。


In [6]:
def train_bpe(
    corpus: list[str], num_merges: int
) -> tuple[BPETokenizer, list[tuple[tuple[str, str], int]]]:
    """コーパスから最頻出ペアの統合を num_merges 回繰り返し、トークナイザと学習履歴を返す。"""
    vocab: set[str] = set()
    for text in corpus:
        vocab.update(text)  # 初期語彙はコーパス中の全文字

    token_lists = [list(text) for text in corpus]
    merge_rules: list[tuple[str, str]] = []
    history: list[tuple[tuple[str, str], int]] = []

    for _ in range(num_merges):
        counter = count_pairs(token_lists)
        if not counter:
            break  # これ以上まとめるペアがない
        best_pair, count = counter.most_common(1)[0]

        merge_rules.append(best_pair)
        vocab.add("".join(best_pair))
        token_lists = [merge_pair(tokens, best_pair) for tokens in token_lists]
        history.append((best_pair, count))

    return BPETokenizer(merge_rules=merge_rules, vocab=vocab), history


tokenizer, history = train_bpe(corpus, num_merges=2)

for step, (pair, count) in enumerate(history, start=1):
    print(f"マージ{step}: ({pair[0]}, {pair[1]}) → {''.join(pair)}（頻度 {count}）")

マージ1: (吾, 輩) → 吾輩（頻度 3）
マージ2: (猫, で) → 猫で（頻度 2）


1 回目に `(吾, 輩)`、2 回目に `(猫, で)` がまとまりました。この 2 つが、学習で得られたマージ規則です。獲得された語彙も見てみましょう。

In [7]:
print("マージ規則:", tokenizer.merge_rules)
print("語彙:", sorted(tokenizer.vocab))
print("語彙サイズ:", len(tokenizer.vocab))

マージ規則: [('吾', '輩'), ('猫', 'で')]
語彙: ['あ', 'す', 'で', 'と', 'は', 'る', '吾', '吾輩', '猫', '猫で', '輩']
語彙サイズ: 11


初期の 9 文字に、マージで生まれた `吾輩` と `猫で` が加わり、語彙サイズは 11 になりました。マージ回数だけ語彙が増えていることが確認できます。

## 学習したトークナイザで分割する

トークン化は学習と違い、最頻ペアの探索はしません。**学習で得たマージ規則を、先頭から順にそのまま適用するだけ**です。この処理はすでに `BPETokenizer.encode` に実装済みなので、呼び出して確かめます。まずは学習コーパスを分割してみましょう。

In [8]:
for text in corpus:
    print(text, "→", tokenizer.encode(text))

吾輩は猫である → ['吾輩', 'は', '猫で', 'あ', 'る']
猫と吾輩 → ['猫', 'と', '吾輩']
吾輩猫です → ['吾輩', '猫で', 'す']


`吾輩は猫である` は `[吾輩, は, 猫で, あ, る]` に分割されます。`ある` はひとまとまりに見えますが、`(あ, る)` は学習中に最頻出ペアにならずマージ規則になっていないため、`あ` と `る` に分かれたままです。

`猫で` はあまり自然なまとまりではありませんが、「猫」と「で」が頻繁に隣り合ったために 1 トークンとして獲得されました。BPE が意味ではなく頻度でサブワードを決めることが表れています。

## マージ回数を変えて粒度を観察する

3 文だけのコーパスでは、まとめられるペアがすぐに尽きてしまいます。もう少し文を増やしたコーパスで、マージを重ねるほど頻度の高い部分文字列がまとまる様子を見てみましょう。

In [11]:
large_corpus = [
    "吾輩は猫である",
    "吾輩は犬である",
    "猫は魚が好きである",
    "犬は肉が好きである",
    "吾輩は魚が好きである",
    "猫も犬も好きである",
    "猫である",
    "犬である",
]

tokenizer_large, history_large = train_bpe(large_corpus, num_merges=12)

for step, (pair, count) in enumerate(history_large, start=1):
    print(f"マージ{step:2d}: ({pair[0]}, {pair[1]}) → {''.join(pair)}（頻度 {count}）")

マージ 1: (で, あ) → であ（頻度 8）
マージ 2: (であ, る) → である（頻度 8）
マージ 3: (好, き) → 好き（頻度 4）
マージ 4: (好き, である) → 好きである（頻度 4）
マージ 5: (吾, 輩) → 吾輩（頻度 3）
マージ 6: (吾輩, は) → 吾輩は（頻度 3）
マージ 7: (が, 好きである) → が好きである（頻度 3）
マージ 8: (猫, である) → 猫である（頻度 2）
マージ 9: (犬, である) → 犬である（頻度 2）
マージ10: (魚, が好きである) → 魚が好きである（頻度 2）
マージ11: (吾輩は, 猫である) → 吾輩は猫である（頻度 1）
マージ12: (吾輩は, 犬である) → 吾輩は犬である（頻度 1）


1 回目に得られる `であ` は、それ自体は意味のまとまりではありません。しかしそれが踏み台になって、2 回目に `である` が生まれます。

また、後半のマージ（11 回目・12 回目）では `吾輩は猫である` のように文まるごとが 1 トークンになっています。今回は、コーパスが小さいためですが、学習データによってはこうした思わぬ癖も語彙に反映されます。
例えば、Wikipedia の記事で語彙を学習すると、「この記事には独自研究が含まれているおそれがあります。」のようなトークンが学習されるかもしれません。

同じ文を、マージ回数の違う複数のトークナイザで分割し、粒度の変化を比べてみます。

In [12]:
sample = "猫は魚が好きである"
for num_merges in [0, 3, 6, 12]:
    tok, _ = train_bpe(large_corpus, num_merges=num_merges)
    print(f"merges={num_merges:2d}: {tok.encode(sample)}")

merges= 0: ['猫', 'は', '魚', 'が', '好', 'き', 'で', 'あ', 'る']
merges= 3: ['猫', 'は', '魚', 'が', '好き', 'である']
merges= 6: ['猫', 'は', '魚', 'が', '好きである']
merges=12: ['猫', 'は', '魚が好きである']


マージ 0 回では完全な文字単位、回数を増やすほどトークンが長くまとまり、12 回では `魚が好きである` が 1 トークンになりました。マージ回数（語彙サイズは初期文字数＋マージ回数）が、分割の粒度を直接コントロールしていることが分かります。

## まとめ

- 学習は「隣接ペアを数える（`count_pairs`）→ 最頻ペアをまとめる（`merge_pair`）」の繰り返しで実装できます。
- トークン化は、学習で得たマージ規則を先頭から順に適用します。

## 実験してみましょう

コーパスをより大規模なものに差し替えると、どんなサブワードが最初にまとまるでしょうか？ コーパスの癖が語彙に表れる様子を観察してみましょう。
